# The 1-day signal

The 1-day short volume ratio (SVR1) was one of the four pre-registered signals.
It is the only one that met the success bar, so this notebook examines it in depth.

**Signal:** `s` = minus the neutralized SVR1, so a positive IC means stocks with *less* off-exchange shorting today outperform tomorrow.
**Target:** open(t+1) to close(t+1), the first session after FINRA publishes day t's file.

Sections 1 and 5 apply the pre-registered criteria and construction.
Everything else is exploratory: it was not specified in advance, and the holdout has already been used.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from shortvol import evaluate as ev, factors, features, finra, prices

pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
df = features.add_all(panel, days)
df["s"] = -ev.neutralize(df, "svr1", ["ret1", "ret5", "log_adv20", "abnormal_volume"])

HOLDOUT = pd.Timestamp("2024-01-02")
df["sample"] = np.where(df.date < HOLDOUT, "development", "holdout")
df["period"] = pd.cut(df.date, pd.to_datetime(["2015-12-31", "2019-12-31", "2021-12-31", "2023-12-31", "2026-12-31"]),
                      labels=["2016-19", "2020-21", "2022-23", "2024-26 (holdout)"])
df["year"] = df.date.dt.year
df["size"] = ev.terciles(df, "log_adv20", ["small", "mid", "large"])
df["off_exchange_tercile"] = ev.terciles(df, "off_exchange_share", ["low", "mid", "high"])
df["day_t"] = np.where(df.ret1 > 0, "up day", "down day")

def ic_table(group):
    return ev.ic_by_group(df, "s", "fwd_oc", group)[["mean_ic", "nw_t", "pct_positive"]]

## 1. The pre-registered success bar
1. Development IC with |Newey-West t| >= 3.
2. The same sign in at least 2 of the 3 development periods.
3. The holdout has the same sign with |t| >= 2.

Four signals were tested on two targets, so eight headline numbers.
A Bonferroni correction for eight tests at 5% raises the bar from |t| = 1.96 to about 2.73.

In [2]:
by_sample = ic_table("sample")
by_period = ic_table("period")
dev, hold = by_sample.loc["development"], by_sample.loc["holdout"]
dev_periods = by_period.iloc[:3]
same_sign = int((np.sign(dev_periods.mean_ic) == np.sign(dev.mean_ic)).sum())
print(f"1. development IC {dev.mean_ic:.4f}, t {dev.nw_t:.2f}: {'PASS' if abs(dev.nw_t) >= 3 else 'FAIL'}")
print(f"2. same sign in {same_sign} of 3 periods: {'PASS' if same_sign >= 2 else 'FAIL'}")
print(f"3. holdout IC {hold.mean_ic:.4f}, t {hold.nw_t:.2f}: {'PASS' if np.sign(hold.mean_ic) == np.sign(dev.mean_ic) and abs(hold.nw_t) >= 2 else 'FAIL'}")
print(f"Bonferroni bar (8 tests): development |t| {abs(dev.nw_t):.2f} vs 2.73")
by_period

1. development IC 0.0045, t 3.47: PASS
2. same sign in 2 of 3 periods: PASS
3. holdout IC 0.0065, t 2.44: PASS
Bonferroni bar (8 tests): development |t| 3.47 vs 2.73


,mean_ic,nw_t,pct_positive
2016-19,0.0088,5.0708,0.5635
2020-21,-0.0029,-1.1233,0.4752
2022-23,0.0034,1.3029,0.5110
2024-26 (holdout),0.0065,2.4373,0.5517


## 2. Stability by year
2021 is the one clearly negative year: the meme-stock period, when retail buying routed through wholesalers dominated off-exchange volume.

In [3]:
ic_table("year")

,mean_ic,nw_t,pct_positive
2016,0.0042,1.1732,0.5420
2017,0.0188,6.2469,0.6175
2018,0.0043,1.1436,0.5219
2019,0.0078,2.6115,0.5714
2020,0.0048,1.3038,0.5375
2021,-0.0107,-3.2825,0.4127
2022,0.0051,1.4901,0.5139
2023,0.0017,0.4265,0.5080
2024,0.0043,1.2038,0.5317
2025,0.0128,2.6813,0.5760


## 3. How long the effect lasts
IC against the open-to-close return 1, 2, 3 and 5 sessions ahead.
The effect is concentrated in the first session, and part of it reverses over the following days, so some of it is temporary price pressure rather than permanent information.

In [4]:
pd.DataFrame({k: ev.summarize_ic(ev.daily_ic(df.assign(y=ev.shift_target(df, "fwd_oc", k)), "s", "y"))
              for k in [1, 2, 3, 5]}).T[["mean_ic", "nw_t"]]

,mean_ic,nw_t
1,0.0050,4.2376
2,-0.0012,-1.0386
3,-0.0030,-2.7991
5,-0.0022,-1.9327


## 4. Where it is strongest
- **Size:** stronger in the largest, most liquid names.
- **Off-exchange share:** stronger where more volume trades off-exchange, where the file covers more of the stock's trading.
- **Day-t direction:** stronger after down days.
  Dealers shorting to fill customer buys (Thesis B) would show up after up days; informed traders adding to bad news (Thesis A) after down days.

In [5]:
pd.concat({g: ic_table(g) for g in ["size", "off_exchange_tercile", "day_t"]})

mean_ic   nw_t  pct_positive
size                 large      0.0072 3.9039        0.5377
                     mid        0.0046 2.6684        0.5163
                     small      0.0026 1.5604        0.5054
off_exchange_tercile high       0.0104 5.8438        0.5448
                     low        0.0015 0.8882        0.5126
                     mid        0.0045 2.6588        0.5148
day_t                down day   0.0063 3.6441        0.5337
                     up day     0.0033 2.0697        0.5234

## 5. Long-short backtest
Long the quintile with the least shorting, short the quintile with the most, equal weight, rebuilt every day at the open and closed at the close.
Gross of costs. Turnover is the daily sum of absolute weight changes (a full flip of both legs is 4).

In [6]:
ls = ev.long_short(df, "s", "fwd_oc")
ls["sample"] = np.where(ls.index < HOLDOUT, "development", "holdout")

def backtest_stats(x):
    r = x.ret.dropna()
    return pd.Series({
        "bps_per_day": r.mean() * 1e4,
        "nw_t": ev.newey_west_t(r),
        "sharpe_gross": r.mean() / r.std() * np.sqrt(252),
        "turnover": x.turnover.mean(),
        "breakeven_bps": r.mean() / x.turnover.mean() * 1e4,
    })

pd.concat({"all": backtest_stats(ls), **{k: backtest_stats(g) for k, g in ls.groupby("sample")}}, axis=1).T

,bps_per_day,nw_t,sharpe_gross,turnover,breakeven_bps
all,1.8455,3.2216,1.0937,2.0529,0.8990
development,1.8353,2.8646,1.1100,2.1117,0.8691
holdout,1.8760,1.5038,1.0503,1.8768,0.9996


## 6. Is it just a known factor?
Daily long-short returns regressed on the Fama-French five factors, momentum and short-term reversal.
The portfolio is formed from day t's file and earns day t+1's session, so it is matched with day t+1's factor returns.
Caveat: the factors are close-to-close returns, while the portfolio holds only open-to-close.

In [7]:
f = factors.load()
next_day = pd.Series(days[1:], index=days[:-1])
r = ls.ret.dropna()
r.index = r.index.map(next_day)
X = sm.add_constant(f.drop(columns="RF").reindex(r.index)).dropna()
fit = sm.OLS(r.loc[X.index], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"alpha {fit.params['const'] * 1e4:.2f} bps/day (t {fit.tvalues['const']:.2f}), R-squared {fit.rsquared:.3f}")
pd.DataFrame({"coef": fit.params, "t": fit.tvalues})

alpha 2.02 bps/day (t 3.72), R-squared 0.100


,coef,t
const,0.0002,3.7210
Mkt-RF,-0.0501,-5.1280
SMB,-0.0533,-5.2125
HML,0.0408,3.5572
RMW,-0.0155,-1.1780
CMA,-0.0197,-1.0499
Mom,0.0356,4.8849
ST_Rev,0.0237,1.6756
